In [1]:
import os
os.chdir(r"C:\Users\chagu\PycharmProjects\Generative AI Chat Bot Project")
print("✅ Working directory:", os.getcwd())

✅ Working directory: C:\Users\chagu\PycharmProjects\Generative AI Chat Bot Project


In [2]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

C:\Users\chagu\AppData\Local\Temp\ipykernel_40500\2763585281.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
c:\Users\chagu\PycharmProjects\Generative AI Chat Bot Project\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [4]:
documents = load_pdf_file("Data/")
print(f"✅ Loaded {len(documents)} pages")

✅ Loaded 638 pages


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [7]:
text_chunks = text_split(documents)
print(f"✅ Total chunks: {len(text_chunks)}")

✅ Total chunks: 4227


In [8]:
extract_data=load_pdf_file(data="Data/")

In [9]:
extract_data = load_pdf_file(data="Data/")
print(f"✅ Pages extracted: {len(extract_data)}")
print(extract_data[0])  # preview first page

✅ Pages extracted: 638
page_content='h Edition 
Ardhendu Sinha Ray 
Abhise Si a a.v 
https://t.me/MBS_MedicalBooksStore' metadata={'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creator': 'PDF-XChange Editor 6.0.317.1', 'creationdate': '2018-09-01T12:35:34-05:00', 'moddate': '2019-10-26T19:16:43+03:00', 'mebooksfree_image': '{10A2F861-FCB1-438D-B23D-FC5BC5445063}', 'source': 'Data\\40mbs_medicalbooksstore_2017_essentials.pdf', 'total_pages': 638, 'page': 0, 'page_label': 'I'}


In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

In [11]:
# Download the embeddings from hugging face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2',)
    return embeddings

In [12]:
embeddings=download_hugging_face_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3089.34it/s]


In [13]:
query_result = embeddings.embed_query("Hello world")
print("length", len(query_result))

length 384


In [14]:
#query_result

In [15]:
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
PINECONE_API_KEY=os.environ.get('OPENAI_API_KEY')

In [17]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index_name = "medicalbot"

if index_name not in [i["name"] for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
else:
    print(f"Index '{index_name}' already exists — skipping creation.")

index = pc.Index(index_name)

Index 'medicalbot' already exists — skipping creation.


In [18]:
import sys
print(sys.executable)

c:\Users\chagu\PycharmProjects\Generative AI Chat Bot Project\.venv312\Scripts\python.exe


In [19]:

#import os
#os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
#os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [20]:
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index
docsearch = PineconeVectorStore.from_documents(
   documents=text_chunks,
   index_name =index_name,
   embedding=embeddings,

)

In [21]:
docsearch

In [22]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [23]:
retriever_docs = retriever.invoke("What is Hypertension?") 

In [24]:
retriever_docs

[Document(id='2c6db3a7-0a30-4314-8612-1738f5ff132b', metadata={'creationdate': '2018-09-01T12:35:34-05:00', 'creator': 'PDF-XChange Editor 6.0.317.1', 'mebooksfree_image': '{10A2F861-FCB1-438D-B23D-FC5BC5445063}', 'moddate': '2019-10-26T19:16:43+03:00', 'page': 61.0, 'page_label': '55', 'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'source': 'Data\\40mbs_medicalbooksstore_2017_essentials.pdf', 'total_pages': 638.0}, page_content='Definition\nElevation of systolic or diastolic blood pressure above the \nnormal range is called hypertension. In 92–94% patients \nno cause could be found out after extensive investigation \nand is called essential hypertension and in the remaining \n6–8% patients hypertension is secondary to diseases of the \nkidney or other endocrine organ and is called secondary \nhypertension.\n • Malignant hypertension —When hypertension causes \npapilledema, deteriorating renal function and encepha-'),
 Document(id='1835abfa-2fa

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4, max_tokens=500)

c:\Users\chagu\PycharmProjects\Generative AI Chat Bot Project\.venv312\Lib\site-packages\IPython\core\interactiveshell.py:3688: UserWarning: WARNING! max_token is not default parameter.
                max_token was transferred to model_kwargs.
                Please confirm that max_token is what you intended.
  if await self.run_code(code, result, async_=asy):


In [28]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
   "You are a medical information assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If the context doesn't contain enough information "
    "to answer, say that you don't know. Do not use outside knowledge "
    "or make assumptions beyond what's given. Use three sentences "
    "maximum and keep the answer concise.\n\n"
    "This is general medical information only, not a diagnosis or "
    "treatment plan. For specific health concerns, remind the user "
    "to consult a licensed healthcare provider. If the question "
    "describes a medical emergency, tell the user to seek emergency "
    "care immediately instead of answering from context."
    "\n\n"
    "{context}"
  

)

prompt = ChatPromptTemplate.from_messages(
  [("system", system_prompt),
   ("human", "{input}"),
   ]
)

In [29]:
question_answer_chain = create_retrieval_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [31]:
response = rag_chain.invoke({"input": "what is Hypertension?"})
print(response["answer"])

ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.